# pydreg — GPU-accelerated dREG on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adamyhe/pydreg/blob/main/notebooks/pydreg_colab.ipynb)

[pydreg](https://github.com/adamyhe/pydreg) detects active transcriptional regulatory elements (promoters and enhancers) from PRO-seq/GRO-seq/ChRO-seq nascent-transcription data. This notebook runs the full pipeline on a Colab GPU.

**Before you start:** make sure this runtime has a GPU attached:
- *Runtime → Change runtime type → T4 GPU*

pydreg will still run on CPU-only, but it will be **much** slower.

## 1. Install pydreg

In [ ]:
!pip install -q pydreg[gpu]

Verify the GPU is detected:

In [ ]:
import subprocess, shutil

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
else:
    print("No GPU detected — pydreg will fall back to CPU (much slower).")
    print("Go to Runtime → Change runtime type → T4 GPU, then re-run.")

from pydreg import backend
print(f"\npydreg scoring backend: {backend.detect_backend()}")

## 2. Get bigWig files

pydreg needs a pair of strand-specific bigWig files — plus-strand and minus-strand read counts (3′-mapped, point-mode, unnormalized). These are the same input files the original dREG expects.

**Option A: Download example data from GEO** — K562 GRO-seq from [Danko et al. 2015](https://doi.org/10.1038/nmeth.3329) ([GSM1480325](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1480325)), the fastest of pydreg's 12 benchmarked libraries (~20–25 min on a Colab T4).

In [ ]:
!wget -q --show-progress -O K562_GROseq_plus.bw \
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1480nnn/GSM1480325/suppl/GSM1480325%5FK562%5FGROseq%5Fplus.bigWig"
!wget -q --show-progress -O K562_GROseq_minus.bw \
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1480nnn/GSM1480325/suppl/GSM1480325%5FK562%5FGROseq%5Fminus.bigWig"

plus_bw  = "K562_GROseq_plus.bw"
minus_bw = "K562_GROseq_minus.bw"

**Option B: Upload from your computer** (run the cell below instead, then select your files in the dialog).

In [ ]:
# from google.colab import files

# print("Select your plus-strand bigWig:")
# uploaded = files.upload()
# plus_bw = list(uploaded.keys())[0]
# print(f"  → {plus_bw}")

# print("\nSelect your minus-strand bigWig:")
# uploaded = files.upload()
# minus_bw = list(uploaded.keys())[0]
# print(f"  → {minus_bw}")

**Option C: Download from a URL** (edit the URLs below and run this cell instead).

In [ ]:
# !wget -q -O plus.bw  "https://your-server.example.com/sample_plus.bw"
# !wget -q -O minus.bw "https://your-server.example.com/sample_minus.bw"
# plus_bw  = "plus.bw"
# minus_bw = "minus.bw"

**Option D: Copy from Google Drive.**

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

# plus_bw  = "/content/drive/MyDrive/path/to/plus.bw"
# minus_bw = "/content/drive/MyDrive/path/to/minus.bw"

## 3. Run pydreg

In [ ]:
import multiprocessing

out_prefix = "output/result"
cores = multiprocessing.cpu_count()

print(f"Plus:   {plus_bw}")
print(f"Minus:  {minus_bw}")
print(f"Output: {out_prefix}.*")
print(f"Cores:  {cores}")

In [ ]:
import os
os.makedirs(os.path.dirname(out_prefix), exist_ok=True)

from pydreg import pipeline

result = pipeline.run(
    plus_bw,
    minus_bw,
    out_prefix,
    cores=cores,
    progress=True,
)

print(f"\nDone — {result['peak_bed'].shape[0]} peaks called.")

## 4. Inspect results

In [ ]:
peaks = result["peak_bed"]
print(f"Peaks called: {len(peaks)}")
print(f"Min score:    {result['min_score']:.4f}")
print()
peaks.head(10)

## 5. Download output files

pydreg writes several output files (see the [README](https://github.com/adamyhe/pydreg#output-files) for the full list):

In [ ]:
import glob

output_files = sorted(glob.glob(f"{out_prefix}.dREG.*"))
for f in output_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

In [ ]:
from google.colab import files

!cd output && zip -q ../pydreg_results.zip result.dREG.*
files.download("pydreg_results.zip")

---

**Citation.** If you use pydreg, please cite:

> He, A. Y., & Danko, C. G. (2026). pydreg: a fast Python package for identifying active cis-regulatory elements from nascent transcription. *bioRxiv*. https://doi.org/10.64898/2026.09.06.745329

And the original dREG papers:

> Danko, C. G., et al. (2015). Identification of active transcriptional regulatory elements from GRO-seq data. *Nature Methods*, 12(5), 433-438.

> Wang, Z., et al. (2018). Identification of regulatory elements from nascent transcription using dREG. *Genome Research*, 29, 293–303.